In [54]:
import os
import re
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from math import log

For each data point, please remove the first four lines, i.e. the lines starting with Newsgroup,
document_id, From, Subject.

In [55]:
def process_data(file_path):
    with open(file_path, 'r',encoding= 'latin1') as file:
        lines = file.readlines()
        content = lines[4:]
        content = ' '.join(content)
        content = content.lower()
        content = re.sub(r'[^a-zA-Z\s]', ' ', content)
        content = ' '.join(content.split())
        return content
    
def get_data(root):
    data_by_class = {}
    for folder, dirs, files in os.walk(root):
        class_name = os.path.basename(folder)

        if folder == root:
            continue
        if class_name not in data_by_class:
            data_by_class[class_name] = []

        for file in files:
           file_path = os.path.join(folder,file)
           content = process_data(file_path = file_path)
           data_by_class[class_name].append(content)

    return data_by_class

In [56]:
root_path = "data/20_newsgroups"
data_by_class = get_data(root_path)

Consider the top 200 most frequent words as stop words and remove them from the vocabulary

In [57]:
def get_word_bag(data_by_class):
    word_bag = Counter()
    for class_name, documents in data_by_class.items():
        for doc in documents:
            words = doc.split()  # Split document into words
            word_bag.update(words)  # Update word frequencies
    return word_bag

def remove_stop_words(data_by_class, stop_words):
    filtered_data_by_class = {}
    for class_name, documents in data_by_class.items():
        filtered_data_by_class[class_name] = []
        for doc in documents:
            # Remove stop words from each document
            filtered_doc = ' '.join([word for word in doc.split() if word not in stop_words])
            filtered_data_by_class[class_name].append(filtered_doc)
    return filtered_data_by_class

In [58]:
word_bag = get_word_bag(data_by_class)
top_200_words = [word for word, _ in word_bag.most_common(200)] #  Consider the top 200 most frequent words as stop words 
filtered_data_by_class = remove_stop_words(data_by_class, top_200_words)

Split data into two groups, and use half data as training data and the other half as testing data. Note:
split the data of each class into two halves.

In [59]:
def split_data(data_by_class):
    train_data = {}
    test_data = {}
    for class_name, documents in data_by_class.items():
        train_docs, test_docs = train_test_split(documents, test_size=0.5, random_state=42)
        train_data[class_name] = train_docs
        test_data[class_name] = test_docs

    return train_data, test_data

In [60]:
train_data, test_data = split_data(filtered_data_by_class)

Now implement the Naive Bayes classifier 

In [61]:
class NaiveBayesClassifier:
    def __init__(self):

        self.word_freq_for_each_class = defaultdict(Counter)
        self.total_number_of_words_per_class = defaultdict(int)
        self.num_of_documents_per_class = defaultdict(int)
        self.vocabulary = set()
        self.total_docs = 0
        self.class_priors = defaultdict(float)

    def train(self,train_data):
        self.total_docs = sum(len(docs) for docs in train_data.values())
            
        for class_name, documents in train_data.items():
            self.num_of_documents_per_class[class_name] = len(documents)
            for doc in documents:
                words = doc.split()
                
                self.total_number_of_words_per_class[class_name] += len(words)

                for word in words:
                    self.word_freq_for_each_class[class_name][word] += 1
                    self.vocabulary.add(word)

            self.class_priors[class_name] = log(self.num_of_documents_per_class[class_name] / self.total_docs)

    def predict(self,doc):
        words = doc.split()
        vocab_size = len(self.vocabulary)
        class_scores = {}

        for class_name in self.word_freq_for_each_class.keys():
            log_prob = self.class_priors[class_name]

            for word in words:
                word_freq = self.word_freq_for_each_class[class_name][word] + 1
                word_prob = log(word_freq / (self.total_number_of_words_per_class[class_name] + vocab_size))
                log_prob += word_prob

            class_scores[class_name] = log_prob
        return max(class_scores, key=class_scores.get)
        
    def evaluate(self, test_data):
        correct = 0
        total = 0
        for class_name, documents in test_data.items():
            for doc in documents:
                prediction = self.predict(doc)
                if prediction == class_name:
                    correct += 1
                total += 1

        accuracy = correct / total
        return accuracy

In [62]:
classifier = NaiveBayesClassifier()
classifier.train(train_data)
accuracy = classifier.evaluate(test_data)
print(f"Naive Bayes Classifier Accuracy: {accuracy:.4f}")

Naive Bayes Classifier Accuracy: 0.8306
